In [1]:
import socket
import json
import time
from pynq import Overlay

ol = Overlay("/home/xilinx/jupyter_notebooks/mlp/mlp.bit")
mlp = ol.mlp_axi_v1_0_MLP_AXI_0
CLASS_NAMES = ["circle", "rectangle", "triangle", "line", "freehand"]

In [2]:
def pack_features_to_regs(features):
    if len(features) != 70:
        raise ValueError(f"Expected 70 features, got {len(features)}")
    byte_vals = [(int(x) + 256) % 256 for x in features]
    regs = []
    for i in range(0, 70, 4):
        chunk = byte_vals[i:i+4]
        while len(chunk) < 4:
            chunk.append(0)
        word = (
            (chunk[0] << 0)  |
            (chunk[1] << 8)  |
            (chunk[2] << 16) |
            (chunk[3] << 24)
        )
        regs.append(word)
    while len(regs) < 18:
        regs.append(0)
    return regs

def mlp_write_features(features):
    regs = pack_features_to_regs(features)
    for i, word in enumerate(regs):
        addr = 0x04 + 4 * i
        mlp.write(addr, word)

def mlp_start_hold():
    mlp.write(0x00, 0x1)

def mlp_stop():
    mlp.write(0x00, 0x0)

def mlp_status():
    return mlp.read(0x00)

def mlp_done():
    return (mlp_status() >> 8) & 0x1

def mlp_predict(features, timeout_s=1.0):
    mlp_write_features(features)
    mlp_start_hold()
    t0 = time.time()
    while mlp_done() == 0:
        if time.time() - t0 > timeout_s:
            mlp_stop()
            raise TimeoutError("Timed out waiting for accelerator")
    status = mlp_status()
    cid = (status >> 9) & 0x7
    label = CLASS_NAMES[cid] if 0 <= cid < len(CLASS_NAMES) else f"unknown({cid})"
    mlp_stop()
    return cid, label, status

In [ ]:
UDP_IP = "0.0.0.0"
UDP_PORT = 5005
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind((UDP_IP, UDP_PORT))
print(f"[udp] listening on {UDP_IP}:{UDP_PORT}")

DEFAULT_REPLY_IP   = "192.168.2.1"
DEFAULT_REPLY_PORT = 5006

while True:
    data, addr = sock.recvfrom(65535)
    try:
        payload = json.loads(data.decode("utf-8"))
    except Exception as e:
        print("[error] invalid JSON:", e)
        continue

    if "features" not in payload:
        print("[error] no features field")
        continue

    features   = payload["features"]
    reply_ip   = payload.get("reply_ip",   DEFAULT_REPLY_IP)
    reply_port = int(payload.get("reply_port", DEFAULT_REPLY_PORT))

    print(f"\n[udp] received {len(features)} features from {addr}")
    print("[udp] first 10 features:", features[:10])

    try:
        cid, label, status = mlp_predict(features)
        reply = {"class_id": cid, "label": label, "status": int(status)}
        sock.sendto(json.dumps(reply).encode("utf-8"), (reply_ip, reply_port))
        print("[fpga] predicted:", label)
        print("[fpga] class_id:", cid)
        print("[fpga] status:", hex(status))
        print(f"[udp] replied to {(reply_ip, reply_port)}")
    except Exception as e:
        sock.sendto(json.dumps({"error": str(e)}).encode("utf-8"), (reply_ip, reply_port))
        print("[error] inference failed:", e)

In [4]:
# Stop Cell 3 before running this (interrupt the kernel)
# This cell runs entirely on the PYNQ — no laptop needed

import math

# ── SW baseline: pure Python integer MLP, mirrors fixed_point_ref.py ──────────
def relu(x):
    return x if x > 0 else 0

def quantize_input(vec, scale=64):
    out = []
    for x in vec:
        val = int(round(x * scale))
        val = max(-128, min(127, val))
        out.append(val)
    return out

def mlp_infer_sw(x_q, params):
    fc1_w = params["fc1_weight"]
    fc1_b = params["fc1_bias"]
    fc2_w = params["fc2_weight"]
    fc2_b = params["fc2_bias"]
    hidden = []
    for j in range(16):
        acc = fc1_b[j]
        for i in range(70):
            acc += x_q[i] * fc1_w[j][i]
        hidden.append(relu(acc))
    outputs = []
    for k in range(5):
        acc = fc2_b[k]
        for j in range(16):
            acc += hidden[j] * fc2_w[k][j]
        outputs.append(acc)
    return max(range(5), key=lambda k: outputs[k])

# ── Predesigned test strokes ───────────────────────────────────────────────────
def make_circle(cx=300, cy=300, r=100, n=60):
    return [[int(cx + r*math.cos(2*math.pi*i/n)),
             int(cy + r*math.sin(2*math.pi*i/n))] for i in range(n+1)]

def make_rectangle(x=200, y=200, w=200, h=150, n=60):
    pts = []
    s = n // 4
    for i in range(s): pts.append([x + int(w*i/s), y])
    for i in range(s): pts.append([x+w, y + int(h*i/s)])
    for i in range(s): pts.append([x+w - int(w*i/s), y+h])
    for i in range(s): pts.append([x, y+h - int(h*i/s)])
    return pts

def make_line(x0=100, y0=300, x1=500, y1=300, n=50):
    return [[int(x0+(x1-x0)*i/(n-1)), int(y0+(y1-y0)*i/(n-1))] for i in range(n)]

TEST_STROKES = {
    "circle":    (make_circle(),    0),
    "rectangle": (make_rectangle(), 1),
    "line":      (make_line(),      3),
}

# ── Load weights for SW baseline ──────────────────────────────────────────────
WEIGHTS_PATH = "/home/xilinx/jupyter_notebooks/mlp/mlp_quantized.json"
with open(WEIGHTS_PATH) as f:
    params = json.load(f)

# ── preprocess must be in the same folder ─────────────────────────────────────
from preprocess import preprocess_to_vector

RUNS = 20

print(f"{'Shape':<12} {'Expected':<12} {'HW result':<12} {'SW result':<12} "
      f"{'HW (ms)':>10}  {'SW (ms)':>10}  {'Speedup':>8}  {'Match':>6}")
print("-" * 90)

for shape_name, (stroke, expected_id) in TEST_STROKES.items():
    # preprocess once — not part of timing for either side
    vec   = preprocess_to_vector(stroke, num_points=32, min_distance=2.0)
    x_q   = quantize_input(vec, scale=64)

    # ── HW timing ─────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    for _ in range(RUNS):
        hw_cid, hw_label, _ = mlp_predict(x_q)
    hw_ms = (time.perf_counter() - t0) / RUNS * 1000

    # ── SW timing ─────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    for _ in range(RUNS):
        sw_cid = mlp_infer_sw(x_q, params)
    sw_ms = (time.perf_counter() - t0) / RUNS * 1000

    sw_label = CLASS_NAMES[sw_cid]
    speedup  = sw_ms / hw_ms
    match    = "YES ✓" if hw_cid == sw_cid else "NO ✗"

    print(f"  {shape_name:<10} {CLASS_NAMES[expected_id]:<12} {hw_label:<12} {sw_label:<12} "
          f"{hw_ms:>10.4f}  {sw_ms:>10.4f}  {speedup:>7.2f}x  {match:>6}")

Shape        Expected     HW result    SW result       HW (ms)     SW (ms)   Speedup   Match
------------------------------------------------------------------------------------------
  circle     circle       circle       circle           1.3356      2.4921     1.87x   YES ✓
  rectangle  rectangle    rectangle    rectangle        1.2567      2.5232     2.01x   YES ✓
  line       line         freehand     freehand         1.0787      2.5428     2.36x   YES ✓
